# LASH: Leakage-Aware Sequential Hybrid Learning

> **A validation-gated selective hybrid pipeline for hourly, direct 24-step electricity-demand forecasting**

This notebook provides a reproducible implementation of LASH using two anonymized benchmark datasets:

| Dataset key | Repository file | Training period | Validation period | Test period |
|---|---|---:|---:|---:|
| `CLUSTER_1` | `Cluster 1.csv` | 2015-03 to 2017-02 | 2017-03 to 2018-08 | 2018-09 to 2020-02 |
| `CLUSTER_2` | `Cluster 2.csv` | 2015-09 to 2016-12 | 2017-01 to 2017-12 | 2018-01 to 2018-12 |

### Model components

1. **Leakage-safe anchor** — daily and weekly demand references with holiday-regime correction.
2. **Sequential residual expert** — causal TCN, feature gating, and a compact GRN decoder.
3. **Linear residual expert** — Ridge regression trained on the same leakage-safe information set.
4. **Validation-gated router** — selects a single expert or a conservative horizon-wise convex blend.

### Reproducibility protocol

- Lookback window: **168 hours**
- Forecast horizon: **24 hours**
- Forecast origin stride: **1 hour**
- Validation split: chronological tuning/calibration partitions with a **23-origin purge gap**
- Test policy: all hyperparameters, epochs, residual gains, and router weights are frozen before one-time test evaluation
- Year-over-year demand features: **excluded**

> **Research note:** The no-year-over-year redesign was informed by earlier experimental observations. Treat the resulting test scores as retrospective evidence unless they are confirmed on an external or future holdout dataset.


## 1. Environment setup and hardware check

The notebook uses the PyTorch build already installed in the active Jupyter kernel. The installation cell intentionally excludes PyTorch to avoid replacing a CUDA-compatible build.

**Recommended workflow**

1. Run with `RUN_MODE = "smoke"` to verify data loading and model wiring.
2. Restart the kernel if packages were upgraded.
3. Change to `RUN_MODE = "paper"` for the complete experiment.


In [ ]:
%pip install -q --upgrade-strategy only-if-needed "numpy>=1.26,<3" "pandas>=2.1,<3" "scikit-learn>=1.4,<2" "optuna>=3.6,<5" "matplotlib>=3.8,<4" "seaborn>=0.13,<1" "joblib>=1.3,<2"


In [ ]:
import copy
import gc
import json
import math
import os
import platform
import random
import sys
import time
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import joblib
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
import sklearn
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid", context="notebook")

print({
    "kernel_python": sys.executable,
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
    "optuna": optuna.__version__,
    "torch": torch.__version__,
    "torch_cuda_build": torch.version.cuda,
    "cuda_available": torch.cuda.is_available(),
    "cudnn": torch.backends.cudnn.version(),
})


## 2. Configuration, repository layout, and fixed evaluation splits

Keep the following three core files in the same repository directory:

```text
LASH_Cluster_Benchmark_GPU.ipynb
Cluster 1.csv
Cluster 2.csv
```

Set the optional `LASH_DATA_ROOT` environment variable when the data files are stored elsewhere. Otherwise, the notebook uses the current working directory.

### Fixed chronological splits

- **Cluster 1:** train 2015-03–2017-02; validation 2017-03–2018-08; test 2018-09–2020-02
- **Cluster 2:** train 2015-09–2016-12; validation 2017-01–2017-12; test 2018-01–2018-12
- **Window:** 168-hour lookback and direct `t+1, ..., t+24` prediction
- **Tuning partition:** first two-thirds of validation
- **Purge:** 23 forecast origins between validation partitions
- **Router calibration:** final one-third of validation
- **Test:** evaluated once after all model-selection decisions are frozen


In [ ]:
# -----------------------------------------------------------------------------
# Experiment configuration
# -----------------------------------------------------------------------------
SEED = 42
LOOKBACK = 168
HORIZON = 24
ORIGIN_STRIDE_HOURS = 1

# Use "smoke" for a fast wiring check and "paper" for the complete experiment.
RUN_MODE = "smoke"

# "benchmark_observed": observed target-hour weather for benchmark comparison.
# "historical_only": lagged weather proxies available from historical records.
WEATHER_MODE = "benchmark_observed"

# Dataset identifiers used throughout logs, outputs, and result dictionaries.
DATASETS_TO_RUN = ["CLUSTER_1", "CLUSTER_2"]

# GPU is recommended. Set True to stop execution when CUDA is unavailable.
REQUIRE_CUDA = False
USE_AMP = True
PIPELINE_VERSION = "lash_cluster_selective_hybrid_v1"

# -----------------------------------------------------------------------------
# Repository paths
# -----------------------------------------------------------------------------
PROJECT_ROOT = Path.cwd().resolve()
environment_root = os.environ.get("LASH_DATA_ROOT")
DATA_ROOT = (
    Path(environment_root).expanduser().resolve()
    if environment_root
    else PROJECT_ROOT
)

if RUN_MODE not in {"smoke", "paper"}:
    raise ValueError("RUN_MODE must be either 'smoke' or 'paper'.")
if WEATHER_MODE not in {"benchmark_observed", "historical_only"}:
    raise ValueError(
        "WEATHER_MODE must be either 'benchmark_observed' or 'historical_only'."
    )

FAST_DEV_RUN = RUN_MODE == "smoke"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = bool(USE_AMP and DEVICE.type == "cuda")

# -----------------------------------------------------------------------------
# Search and training controls
# -----------------------------------------------------------------------------
N_TRIALS_SEQ = 3 if FAST_DEV_RUN else 20
MAX_EPOCHS = 10 if FAST_DEV_RUN else 80
EARLY_STOPPING_PATIENCE = 3 if FAST_DEV_RUN else 10
SMOKE_MAX_ORIGINS_PER_SPLIT = 512
VALIDATION_TUNE_FRACTION = 2.0 / 3.0
VALIDATION_PURGE_ORIGINS = HORIZON - 1
RIDGE_ALPHA_GRID = [1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0, 1e3, 1e4]
ROUTER_WEIGHT_GRID = np.linspace(0.0, 1.0, 21)
MIN_HYBRID_IMPROVEMENT = 0.005  # Selection-score percentage points.

RUN_TAG = (
    f"{PIPELINE_VERSION}_{RUN_MODE}_{WEATHER_MODE}"
    f"_trials{N_TRIALS_SEQ}_ep{MAX_EPOCHS}"
)
DATA_DIR_CANDIDATES = [DATA_ROOT, PROJECT_ROOT]
DATA_DIR_CANDIDATES = list(dict.fromkeys(DATA_DIR_CANDIDATES))

RESULT_DIR = PROJECT_ROOT / "outputs" / "LASH" / RUN_TAG
OPTUNA_DIR = RESULT_DIR / "optuna"
MODEL_DIR = RESULT_DIR / "models"
FIGURE_DIR = RESULT_DIR / "figures"
for directory in [RESULT_DIR, OPTUNA_DIR, MODEL_DIR, FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 600,
    "savefig.facecolor": "white",
    "font.size": 11,
})


@dataclass(frozen=True)
class DatasetSpec:
    """Immutable metadata for one benchmark cluster."""

    name: str
    filename: str
    timestamp_format: str
    timestamp_shift_hours: int
    train_start: str
    train_end: str
    val_start: str
    val_end: str
    test_start: str
    test_end: str
    demand_unit: str


SPECS = {
    "CLUSTER_1": DatasetSpec(
        name="Cluster 1",
        filename="Cluster 1.csv",
        timestamp_format="date_string",
        timestamp_shift_hours=-1,
        train_start="2015-03-01",
        train_end="2017-03-01",
        val_start="2017-03-01",
        val_end="2018-09-01",
        test_start="2018-09-01",
        test_end="2020-03-01",
        demand_unit="Original public-data unit",
    ),
    "CLUSTER_2": DatasetSpec(
        name="Cluster 2",
        filename="Cluster 2.csv",
        timestamp_format="components",
        timestamp_shift_hours=0,
        train_start="2015-09-01",
        train_end="2017-01-01",
        val_start="2017-01-01",
        val_end="2018-01-01",
        test_start="2018-01-01",
        test_end="2019-01-01",
        demand_unit="Original public-data unit",
    ),
}


def set_seed(seed: int = SEED) -> None:
    """Configure reproducible random states across Python, NumPy, and PyTorch."""

    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    if hasattr(torch, "set_float32_matmul_precision"):
        torch.set_float32_matmul_precision("high")


set_seed()
if REQUIRE_CUDA and not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is required by the current configuration, but no CUDA device is "
        "available in the active Jupyter kernel."
    )

runtime_info = {
    "device": str(DEVICE),
    "AMP_enabled": AMP_ENABLED,
    "run_mode": RUN_MODE,
    "data_root": str(DATA_ROOT),
    "result_dir": str(RESULT_DIR),
}
if torch.cuda.is_available():
    properties = torch.cuda.get_device_properties(0)
    runtime_info.update({
        "gpu_name": torch.cuda.get_device_name(0),
        "compute_capability": torch.cuda.get_device_capability(0),
        "gpu_vram_GB": round(properties.total_memory / 1024**3, 2),
        "cuda_probe": (torch.ones(1024, device="cuda") * 2).sum().item(),
    })

display(runtime_info)
if FAST_DEV_RUN:
    print(
        "[SMOKE MODE] Results are intended for pipeline verification only, "
        "not for paper tables."
    )


### Automatic CSV discovery

The loader prioritizes exact repository filenames and also tolerates browser-generated duplicate suffixes such as `(1)` or `(2)`.


In [ ]:
def _strip_browser_duplicate_suffix(filename: str) -> str:
    """Normalize browser-generated duplicate suffixes such as ``(1)``."""

    path = Path(filename)
    stem = path.stem.rstrip()
    if stem.endswith(")") and "(" in stem:
        prefix, suffix = stem.rsplit("(", 1)
        if suffix[:-1].isdigit():
            stem = prefix.rstrip()
    return f"{stem}{path.suffix}".casefold()


def _find_data_path(filename: str) -> Optional[Path]:
    """Resolve one CSV from the configured repository search paths."""

    # 1) Always prioritize an exact filename match.
    for root in DATA_DIR_CANDIDATES:
        exact = root / filename
        if exact.exists():
            return exact

    # 2) Tolerate browser duplicate suffixes and case differences only.
    canonical = _strip_browser_duplicate_suffix(filename)
    matches = []
    for root in DATA_DIR_CANDIDATES:
        if root.exists():
            matches.extend(
                p for p in root.glob("*.csv")
                if _strip_browser_duplicate_suffix(p.name) == canonical
            )
    matches = sorted(set(matches), key=lambda p: (len(p.name), p.name.casefold()))
    if matches:
        if len(matches) > 1:
            print(f"[FILE MATCH] Multiple candidates found for {filename}; using {matches[0]}: {matches}")
        return matches[0]
    return None


def resolve_data_path(filename: str) -> Path:
    path = _find_data_path(filename)
    if path is not None:
        return path
    searched = [str(root / filename) for root in DATA_DIR_CANDIDATES]
    raise FileNotFoundError(f"Could not find {filename}. Searched: {searched}")


def ensure_input_csvs() -> Dict[str, Path]:
    """Validate and return both cluster CSV paths before preprocessing."""

    if not DATA_ROOT.exists():
        raise FileNotFoundError(f"DATA_ROOT does not exist: {DATA_ROOT}")

    resolved = {}
    still_missing = []
    for name, spec in SPECS.items():
        path = _find_data_path(spec.filename)
        if path is None:
            still_missing.append(spec.filename)
        else:
            resolved[name] = path
    if still_missing:
        raise FileNotFoundError(
            "Missing required CSV files: " + ", ".join(still_missing)
            + f"\nPlace both files in {DATA_ROOT} and rerun this cell."
        )
    print("Resolved input files:", {name: str(path) for name, path in resolved.items()})
    return resolved


INPUT_PATHS = ensure_input_csvs()


## 3. Normalize both datasets to one schema

Cluster 1 provides a `Date` column, whereas Cluster 2 provides `Year`, `Month`, `Day`, and `Hour` columns. Both inputs are converted to the same hourly `DatetimeIndex` and feature schema.

Precomputed lag columns in the source files are used only for integrity checks. All model inputs are rebuilt inside this notebook.


In [ ]:
def _parse_timestamp(raw: pd.DataFrame, spec: DatasetSpec) -> pd.DatetimeIndex:
    if spec.timestamp_format == "date_string":
        ts = pd.to_datetime(raw["Date"], errors="raise")
    elif spec.timestamp_format == "components":
        year = raw["Year"].astype(int)
        year = np.where(year < 100, year + 2000, year)
        ts = pd.to_datetime({
            "year": year,
            "month": raw["Month"].astype(int),
            "day": raw["Day"].astype(int),
            "hour": raw["Hour"].astype(int),
        }, errors="raise")
    else:
        raise ValueError(spec.timestamp_format)
    ts = pd.DatetimeIndex(ts) + pd.Timedelta(hours=spec.timestamp_shift_hours)
    return ts


def load_common_frame(spec: DatasetSpec) -> Tuple[pd.DataFrame, pd.DataFrame]:
    path = resolve_data_path(spec.filename)
    raw = pd.read_csv(path, encoding="utf-8-sig")
    required = {"Holi", "Temp", "Humi", "WS", "Consumption"}
    missing = required.difference(raw.columns)
    if missing:
        raise ValueError(f"{spec.name}: missing required columns {sorted(missing)}")

    ts = _parse_timestamp(raw, spec)
    if ts.has_duplicates:
        dup = ts[ts.duplicated()].unique()[:5]
        raise ValueError(f"{spec.name}: duplicate timestamps {dup}")

    frame = pd.DataFrame(index=ts)
    frame.index.name = "timestamp"
    frame["Consumption"] = pd.to_numeric(raw["Consumption"], errors="raise").to_numpy()
    for col in ["Holi", "Temp", "Humi", "WS"]:
        frame[col] = pd.to_numeric(raw[col], errors="raise").to_numpy()
    frame["Holi_source"] = frame["Holi"].astype(int)

    # Rebuild a shared calendar representation for both clusters.
    frame["Year"] = frame.index.year
    frame["Month"] = frame.index.month
    frame["Day"] = frame.index.day
    frame["Hour"] = frame.index.hour
    frame["Weekday"] = frame.index.dayofweek  # Monday=0
    # Cluster 1 stores 00:00 as an interval end; shifting by -1 hour maps it to 23:00.
    # Use the daily modal holiday label to enforce one calendar-day regime per 24 rows.
    day_key = pd.Series(frame.index.normalize(), index=frame.index)
    frame["Holi"] = frame.groupby(day_key)["Holi_source"].transform(
        lambda s: int(s.mode().iloc[0])
    ).astype(int)

    frame = frame.sort_index()
    expected = pd.date_range(frame.index.min(), frame.index.max(), freq="h")
    if not frame.index.equals(expected):
        missing_ts = expected.difference(frame.index)
        raise ValueError(f"{spec.name}: {len(missing_ts)} missing hourly timestamps; examples: {missing_ts[:5].tolist()}")
    if frame.isna().any().any():
        raise ValueError(f"{spec.name}: source data contain missing values: {frame.isna().sum().to_dict()}")

    # Use source-provided lags only for integrity auditing, never as model inputs.
    audit = []
    for supplied, base, lag in [
        ("Cons_1", "Consumption", 24), ("Cons_7", "Consumption", 168),
        ("Holi_1", "Holi_source", 24), ("Holi_7", "Holi_source", 168),
    ]:
        if supplied in raw.columns:
            supplied_s = pd.Series(pd.to_numeric(raw[supplied]).to_numpy(), index=frame.index)
            rebuilt = frame[base].shift(lag)
            valid = rebuilt.notna()
            audit.append({
                "dataset": spec.name,
                "column": supplied,
                "definition": f"{base}.shift({lag})",
                "MAE": float(np.mean(np.abs(supplied_s[valid] - rebuilt[valid]))),
                "max_abs_error": float(np.max(np.abs(supplied_s[valid] - rebuilt[valid]))),
            })
    audit_df = pd.DataFrame(audit)
    return frame, audit_df


dataset_frames = {}
lag_audits = []
for dataset_name, spec in SPECS.items():
    dataset_frames[dataset_name], audit = load_common_frame(spec)
    lag_audits.append(audit)

display(pd.concat(lag_audits, ignore_index=True))
display(pd.DataFrame([
    {
        "dataset": SPECS[name].name,
        "dataset_key": name,
        "rows": len(df),
        "start": df.index.min(),
        "end": df.index.max(),
        "demand_min": df["Consumption"].min(),
        "demand_mean": df["Consumption"].mean(),
        "demand_max": df["Consumption"].max(),
    }
    for name, df in dataset_frames.items()
]))


## 4. Shared leakage-safe feature engineering

The pipeline retains calendar features, phase-shifted cyclical encodings, nonlinear weather indices, holiday-aware daily and weekly lags, and routine-based historical averages.

It does **not** generate annual lags, year-over-year proxies, annual scaling terms, or annual anchor components. Every future demand-derived input is at least 24 hours older than its target timestamp and is therefore available at the forecast origin.


In [ ]:
CALENDAR_COLS = [
    "hour_sin", "hour_cos", "hour_shift_sin", "hour_shift_cos",
    "dow_sin", "dow_cos", "month_sin", "month_cos", "doy_sin", "doy_cos",
    "Holi", "weekend", "holi_hour_interaction",
]
WEATHER_COLS = [
    "Temp", "Humi", "WS", "THI_calc", "WCT_calc",
    "HDD18", "CDD18", "HDD18_sq", "CDD18_sq", "temp_sq", "temp_humi",
]
SAFE_DEMAND_COLS = [
    "cons_lag24_safe", "cons_lag48", "cons_lag168_safe", "cons_lag336",
    "Cons_avg_same_type_7", "recent_mean_24", "recent_mean_168",
    "recent_std_168", "anchor",
]


def _cyclic(values: np.ndarray, period: float) -> Tuple[np.ndarray, np.ndarray]:
    angle = 2.0 * np.pi * np.asarray(values, dtype=float) / period
    return np.sin(angle), np.cos(angle)


def add_common_causal_features(frame: pd.DataFrame) -> pd.DataFrame:
    x = frame.copy()
    idx = x.index

    x["hour_sin"], x["hour_cos"] = _cyclic(idx.hour, 24)
    x["hour_shift_sin"], x["hour_shift_cos"] = _cyclic((idx.hour - 8) % 24, 24)
    x["dow_sin"], x["dow_cos"] = _cyclic(idx.dayofweek, 7)
    x["month_sin"], x["month_cos"] = _cyclic(idx.month - 1, 12)
    x["doy_sin"], x["doy_cos"] = _cyclic(idx.dayofyear - 1, 366)
    x["weekend"] = (idx.dayofweek >= 5).astype(int)
    x["holi_hour_interaction"] = x["Holi"] * x["hour_shift_cos"]

    x["THI_calc"] = (1.8 * x["Temp"] + 32) - (
        (0.55 - 0.0055 * x["Humi"]) * (1.8 * x["Temp"] - 26)
    )
    ws_nonnegative = x["WS"].clip(lower=0)
    x["WCT_calc"] = (
        13.12 + 0.6215 * x["Temp"] - 11.37 * np.power(ws_nonnegative, 0.16)
        + 0.3965 * x["Temp"] * np.power(ws_nonnegative, 0.16)
    )
    x["HDD18"] = (18.0 - x["Temp"]).clip(lower=0)
    x["CDD18"] = (x["Temp"] - 18.0).clip(lower=0)
    x["HDD18_sq"] = x["HDD18"] ** 2
    x["CDD18_sq"] = x["CDD18"] ** 2
    x["temp_sq"] = x["Temp"] ** 2
    x["temp_humi"] = x["Temp"] * x["Humi"]

    y = x["Consumption"]
    for lag in [24, 48, 168, 336]:
        x[f"cons_lag{lag}"] = y.shift(lag)
    x["holi_lag24"] = x["Holi"].shift(24)
    x["holi_lag168"] = x["Holi"].shift(168)

    # Use only the seven previous observations from the same hour and holiday regime.
    x["Cons_avg_same_type_7"] = x.groupby(
        ["Hour", "Holi"], sort=False
    )["Consumption"].transform(lambda s: s.shift(1).rolling(7, min_periods=7).mean())

    # Even at horizon 24, no demand-derived feature occurs after the forecast origin.
    x["recent_mean_24"] = y.shift(24).rolling(24, min_periods=12).mean()
    x["recent_mean_168"] = y.shift(24).rolling(168, min_periods=48).mean()
    x["recent_std_168"] = y.shift(24).rolling(168, min_periods=48).std()

    fallback = x["Cons_avg_same_type_7"].fillna(x["recent_mean_168"])
    x["cons_lag24_safe"] = x["cons_lag24"].where(x["holi_lag24"] == x["Holi"], fallback)
    x["cons_lag168_safe"] = x["cons_lag168"].where(x["holi_lag168"] == x["Holi"], fallback)

    # No-YoY anchor: renormalize only the available daily, weekly, and routine terms.
    candidates = [
        # Preserve the previously validated component ratios. Although they sum to 0.85,
        # the denominator renormalizes available components and preserves the output scale.
        ("cons_lag24_safe", 0.35),
        ("cons_lag168_safe", 0.30),
        ("Cons_avg_same_type_7", 0.20),
    ]
    numerator = np.zeros(len(x), dtype=float)
    denominator = np.zeros(len(x), dtype=float)
    for col, weight in candidates:
        values = x[col].to_numpy(dtype=float)
        valid = np.isfinite(values)
        numerator += np.where(valid, values * weight, 0.0)
        denominator += valid * weight
    x["anchor"] = np.divide(
        numerator, denominator,
        out=x["recent_mean_168"].to_numpy(dtype=float).copy(),
        where=denominator > 0,
    )

    # In historical_only mode, replace target-hour weather with lagged weather proxies.
    for col in ["Temp", "Humi", "WS", "THI_calc", "WCT_calc", "HDD18", "CDD18"]:
        x[f"{col}_lag24"] = x[col].shift(24)
        x[f"{col}_lag168"] = x[col].shift(168)
    return x


feature_frames = {
    name: add_common_causal_features(frame)
    for name, frame in dataset_frames.items()
}

assert not any("yoy" in col.casefold() for col in SAFE_DEMAND_COLS)
for name, df in feature_frames.items():
    generated_yoy = [col for col in df.columns if "yoy" in col.casefold()]
    assert not generated_yoy, (name, generated_yoy)
    target_hours = pd.date_range(
        df.index.min().normalize() + pd.Timedelta(days=8), periods=24, freq="h"
    )
    origin = target_hours[0] - pd.Timedelta(hours=1)
    assert ((target_hours - pd.Timedelta(hours=24)) <= origin).all()
print("No-YoY feature audit: PASS | Causal lag invariant: PASS")


## 5. Build 168-to-24 windows and purged validation partitions

Each sample contains:

- a 168-hour historical sequence;
- 24 hours of forecast-time covariates;
- a leakage-safe anchor trajectory; and
- a 24-hour target trajectory.

Validation is split chronologically into tuning and router-calibration partitions. A 23-origin purge gap prevents overlapping 24-hour target windows across the two partitions.


In [ ]:
PAST_COLS = ["Consumption", *CALENDAR_COLS, *WEATHER_COLS]

FUTURE_BASE_COLS = [
    *CALENDAR_COLS,
    "cons_lag24_safe", "cons_lag48", "cons_lag168_safe", "cons_lag336",
    "Cons_avg_same_type_7", "recent_mean_24", "recent_mean_168",
    "recent_std_168", "anchor",
]
if WEATHER_MODE == "benchmark_observed":
    FUTURE_FRAME_COLS = FUTURE_BASE_COLS + WEATHER_COLS
elif WEATHER_MODE == "historical_only":
    FUTURE_FRAME_COLS = FUTURE_BASE_COLS + [
        f"{col}_lag{lag}"
        for col in ["Temp", "Humi", "WS", "THI_calc", "WCT_calc", "HDD18", "CDD18"]
        for lag in [24, 168]
    ]
else:
    raise ValueError(WEATHER_MODE)
FUTURE_COLS = FUTURE_FRAME_COLS + ["lead_hour"]
assert not any("yoy" in col.casefold() for col in [*PAST_COLS, *FUTURE_COLS])


@dataclass
class WindowBundle:
    past: np.ndarray
    future: np.ndarray
    anchor: np.ndarray
    y: np.ndarray
    forecast_origin: np.ndarray
    target_time: np.ndarray
    split: str
    past_cols: List[str]
    future_cols: List[str]

    def __len__(self) -> int:
        return int(self.y.shape[0])


def _split_bounds(spec: DatasetSpec, split: str) -> Tuple[pd.Timestamp, pd.Timestamp]:
    if split == "train":
        return pd.Timestamp(spec.train_start), pd.Timestamp(spec.train_end)
    if split == "val":
        return pd.Timestamp(spec.val_start), pd.Timestamp(spec.val_end)
    if split == "test":
        return pd.Timestamp(spec.test_start), pd.Timestamp(spec.test_end)
    raise ValueError(split)


def build_windows(frame: pd.DataFrame, spec: DatasetSpec, split: str) -> WindowBundle:
    start, end = _split_bounds(spec, split)
    target_starts = pd.date_range(
        start, end - pd.Timedelta(hours=HORIZON), freq=f"{ORIGIN_STRIDE_HOURS}h"
    )
    target_pos = frame.index.get_indexer(target_starts)
    target_pos = target_pos[(target_pos >= LOOKBACK) & (target_pos + HORIZON <= len(frame))]
    if len(target_pos) == 0:
        raise ValueError(f"{spec.name}-{split}: no forecasting windows were generated.")

    # Smoke mode samples evenly across each chronological split to reduce memory
    # while preserving broad temporal coverage for an end-to-end wiring check.
    if FAST_DEV_RUN and len(target_pos) > SMOKE_MAX_ORIGINS_PER_SPLIT:
        keep = np.linspace(
            0, len(target_pos) - 1, SMOKE_MAX_ORIGINS_PER_SPLIT, dtype=int
        )
        target_pos = target_pos[keep]

    past_idx = target_pos[:, None] + np.arange(-LOOKBACK, 0)[None, :]
    future_idx = target_pos[:, None] + np.arange(0, HORIZON)[None, :]
    past_values = frame[PAST_COLS].to_numpy(dtype=np.float32)
    future_values = frame[FUTURE_FRAME_COLS].to_numpy(dtype=np.float32)
    anchor_values = frame["anchor"].to_numpy(dtype=np.float32)
    target_values = frame["Consumption"].to_numpy(dtype=np.float32)

    past = past_values[past_idx]
    future = future_values[future_idx]
    lead_hour = np.broadcast_to(
        np.arange(1, HORIZON + 1, dtype=np.float32)[None, :, None],
        (len(target_pos), HORIZON, 1),
    )
    future = np.concatenate([future, lead_hour], axis=-1)
    anchor = anchor_values[future_idx]
    target = target_values[future_idx]
    valid = (
        np.isfinite(past).all(axis=(1, 2))
        & np.isfinite(anchor).all(axis=1)
        & np.isfinite(target).all(axis=1)
    )
    if not valid.all():
        past, future = past[valid], future[valid]
        anchor, target = anchor[valid], target[valid]
        target_pos, future_idx = target_pos[valid], future_idx[valid]

    bundle = WindowBundle(
        past=past, future=future, anchor=anchor, y=target,
        forecast_origin=frame.index.to_numpy()[target_pos - 1],
        target_time=frame.index.to_numpy()[future_idx], split=split,
        past_cols=list(PAST_COLS), future_cols=list(FUTURE_COLS),
    )
    origins = pd.to_datetime(bundle.forecast_origin)
    assert np.all(pd.to_datetime(bundle.target_time[:, 0]) == origins + pd.Timedelta(hours=1))
    for h in range(1, HORIZON + 1):
        assert np.all(origins + pd.Timedelta(hours=h - 24) <= origins)
    return bundle


def take_bundle(bundle: WindowBundle, indices: Sequence[int], split: Optional[str] = None) -> WindowBundle:
    idx = np.asarray(indices, dtype=int)
    selector = idx
    if len(idx) and np.array_equal(idx, np.arange(idx[0], idx[0] + len(idx))):
        selector = slice(int(idx[0]), int(idx[-1]) + 1)
    return WindowBundle(
        past=bundle.past[selector], future=bundle.future[selector],
        anchor=bundle.anchor[selector], y=bundle.y[selector],
        forecast_origin=bundle.forecast_origin[selector],
        target_time=bundle.target_time[selector], split=split or bundle.split,
        past_cols=bundle.past_cols, future_cols=bundle.future_cols,
    )


def concat_bundles(*bundles: WindowBundle, split: str) -> WindowBundle:
    first = bundles[0]
    merged = WindowBundle(
        past=np.concatenate([b.past for b in bundles]),
        future=np.concatenate([b.future for b in bundles]),
        anchor=np.concatenate([b.anchor for b in bundles]),
        y=np.concatenate([b.y for b in bundles]),
        forecast_origin=np.concatenate([b.forecast_origin for b in bundles]),
        target_time=np.concatenate([b.target_time for b in bundles]),
        split=split, past_cols=first.past_cols, future_cols=first.future_cols,
    )
    order = np.argsort(merged.forecast_origin)
    return merged if np.array_equal(order, np.arange(len(order))) else take_bundle(merged, order, split)


def split_validation_bundle(val: WindowBundle) -> Tuple[WindowBundle, WindowBundle]:
    cut = int(math.floor(len(val) * VALIDATION_TUNE_FRACTION))
    calibration_start = cut + VALIDATION_PURGE_ORIGINS
    if cut < 2 * HORIZON or len(val) - calibration_start < 2 * HORIZON:
        raise ValueError("The validation period is too short for tune/calibration splitting.")
    tune = take_bundle(val, np.arange(0, cut), split="val_tune")
    calibration = take_bundle(
        val, np.arange(calibration_start, len(val)), split="val_calibration"
    )
    assert pd.Timestamp(tune.target_time.max()) < pd.Timestamp(calibration.target_time.min())
    return tune, calibration


def build_dataset_bundles(name: str) -> Dict[str, WindowBundle]:
    spec = SPECS[name]
    bundles = {
        split: build_windows(feature_frames[name], spec, split)
        for split in ["train", "val", "test"]
    }
    assert bundles["train"].target_time.max() < bundles["val"].target_time.min()
    assert bundles["val"].target_time.max() < bundles["test"].target_time.min()
    val_tune, val_cal = split_validation_bundle(bundles["val"])
    display(pd.DataFrame([
        {
            "dataset": spec.name, "dataset_key": name, "split": split, "forecast_origins": len(bundle),
            "origin_horizon_pairs": bundle.y.size,
            "first_target": pd.Timestamp(bundle.target_time.min()),
            "last_target": pd.Timestamp(bundle.target_time.max()),
        }
        for split, bundle in [
            ("train", bundles["train"]), ("val_tune", val_tune),
            ("purge", None), ("val_calibration", val_cal),
            ("test", bundles["test"]),
        ] if bundle is not None
    ]))
    return bundles


print("past/future dimensions:", len(PAST_COLS), len(FUTURE_COLS))


## 6. Metrics and train-only preprocessing

The evaluation reports MAPE, CVRMSE, and NMAE. The Optuna selection score is the arithmetic mean of these three percentage metrics.

Imputers and scalers are fitted on training data during tuning and on train-plus-validation data during the final refit. Test observations are never used to fit preprocessing objects.


In [ ]:
def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    y_true = np.asarray(y_true, dtype=float).reshape(-1)
    y_pred = np.asarray(y_pred, dtype=float).reshape(-1)
    if y_true.shape != y_pred.shape:
        raise ValueError((y_true.shape, y_pred.shape))
    error = y_true - y_pred
    nonzero = np.abs(y_true) > 1e-12
    mape = np.mean(np.abs(error[nonzero] / y_true[nonzero])) * 100.0 if nonzero.any() else np.nan
    mean_y = float(np.mean(y_true))
    if abs(mean_y) <= 1e-12:
        raise ValueError("The mean of the actual values is zero.")
    cvrmse = np.sqrt(np.mean(error ** 2)) / mean_y * 100.0
    nmae = np.mean(np.abs(error)) / mean_y * 100.0
    return {
        "MAPE": float(mape), "CVRMSE": float(cvrmse), "NMAE": float(nmae),
        "selection_score": float(np.nanmean([mape, cvrmse, nmae])),
        "MAPE_excluded_zero_count": int((~nonzero).sum()),
    }


def per_horizon_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> pd.DataFrame:
    return pd.DataFrame([
        {"horizon": h + 1, **regression_metrics(y_true[:, h], y_pred[:, h])}
        for h in range(HORIZON)
    ])


@dataclass
class DLPreprocessor:
    past_imputer: SimpleImputer
    past_scaler: StandardScaler
    future_imputer: SimpleImputer
    future_scaler: StandardScaler
    target_scaler: StandardScaler


def fit_dl_preprocessor(bundle: WindowBundle) -> DLPreprocessor:
    p = bundle.past.reshape(-1, bundle.past.shape[-1])
    f = bundle.future.reshape(-1, bundle.future.shape[-1])
    residual = (bundle.y - bundle.anchor).reshape(-1, 1)
    p_imp = SimpleImputer(strategy="median", keep_empty_features=True).fit(p)
    p_scaler = StandardScaler().fit(p_imp.transform(p))
    f_imp = SimpleImputer(strategy="median", keep_empty_features=True).fit(f)
    f_scaler = StandardScaler().fit(f_imp.transform(f))
    target_scaler = StandardScaler().fit(residual)
    return DLPreprocessor(p_imp, p_scaler, f_imp, f_scaler, target_scaler)


def transform_dl_inputs(bundle: WindowBundle, prep: DLPreprocessor):
    n = len(bundle)
    p = bundle.past.reshape(-1, bundle.past.shape[-1])
    f = bundle.future.reshape(-1, bundle.future.shape[-1])
    p = prep.past_scaler.transform(prep.past_imputer.transform(p)).reshape(n, LOOKBACK, -1)
    f = prep.future_scaler.transform(prep.future_imputer.transform(f)).reshape(n, HORIZON, -1)
    return p.astype(np.float32), f.astype(np.float32)


def transform_for_dl(bundle: WindowBundle, prep: DLPreprocessor):
    p, f = transform_dl_inputs(bundle, prep)
    residual = prep.target_scaler.transform(
        (bundle.y - bundle.anchor).reshape(-1, 1)
    ).reshape(len(bundle), HORIZON)
    return p, f, residual.astype(np.float32)


class NumpySequenceDataset(Dataset):
    def __init__(self, past: np.ndarray, future: np.ndarray, target: np.ndarray):
        self.past = torch.from_numpy(past)
        self.future = torch.from_numpy(future)
        self.target = torch.from_numpy(target)

    def __len__(self):
        return len(self.target)

    def __getitem__(self, idx):
        return self.past[idx], self.future[idx], self.target[idx]


## 7. Sequential expert: causal TCN, feature gates, and GRN decoder

The sequential expert uses depthwise causal dilated convolutions to capture local and multi-scale temporal patterns over the 168-hour history. Scalar temporal pooling summarizes long-range context, feature gates regulate past and future inputs, and a compact GRN decoder predicts all 24 anchor residuals jointly.

The architecture intentionally avoids bidirectional recurrence and large multi-head attention blocks to preserve computational efficiency.


In [ ]:
class Chomp1d(nn.Module):
    def __init__(self, size: int):
        super().__init__()
        self.size = int(size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x[:, :, :-self.size].contiguous() if self.size > 0 else x


class LiteTemporalBlock(nn.Module):
    """Depthwise causal convolution + pointwise GLU + residual LayerNorm."""
    def __init__(self, d_model: int, kernel_size: int, dilation: int, dropout: float):
        super().__init__()
        padding = (kernel_size - 1) * dilation
        self.depthwise = nn.Conv1d(
            d_model, d_model, kernel_size,
            padding=padding, dilation=dilation, groups=d_model,
        )
        self.chomp = Chomp1d(padding)
        self.pointwise = nn.Conv1d(d_model, 2 * d_model, kernel_size=1)
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        z = x.transpose(1, 2)
        z = self.chomp(self.depthwise(z))
        z = F.glu(self.pointwise(z), dim=1).transpose(1, 2)
        return self.norm(x + self.dropout(z))


class GatedResidualNetwork(nn.Module):
    def __init__(self, d_model: int, dropout: float):
        super().__init__()
        self.candidate = nn.Sequential(
            nn.Linear(d_model, 2 * d_model), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(2 * d_model, d_model),
        )
        self.gate = nn.Linear(d_model, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.norm(x + torch.sigmoid(self.gate(x)) * self.candidate(x))


class LASHSequentialNet(nn.Module):
    def __init__(
        self,
        n_past_features: int,
        n_future_features: int,
        d_model: int,
        tcn_layers: int,
        kernel_size: int,
        dropout: float,
    ):
        super().__init__()
        self.n_past_features = n_past_features
        self.n_future_features = n_future_features
        gate_hidden = max(16, n_past_features)
        self.past_feature_gate = nn.Sequential(
            nn.Linear(n_past_features, gate_hidden), nn.GELU(),
            nn.Linear(gate_hidden, n_past_features),
        )
        self.future_feature_gate = nn.Sequential(
            nn.LayerNorm(n_future_features),
            nn.Linear(n_future_features, n_future_features),
        )
        self.past_projection = nn.Linear(n_past_features, d_model)
        self.temporal_blocks = nn.ModuleList([
            LiteTemporalBlock(d_model, kernel_size, 2 ** layer, dropout)
            for layer in range(tcn_layers)
        ])
        self.temporal_score = nn.Sequential(
            nn.Linear(d_model, max(8, d_model // 2)), nn.Tanh(),
            nn.Linear(max(8, d_model // 2), 1),
        )
        self.context_mix = nn.Sequential(
            nn.Linear(2 * d_model, d_model), nn.GELU(), nn.LayerNorm(d_model),
        )
        self.future_projection = nn.Sequential(
            nn.Linear(n_future_features, d_model), nn.GELU(), nn.Dropout(dropout),
        )
        self.horizon_embedding = nn.Parameter(torch.zeros(1, HORIZON, d_model))
        nn.init.normal_(self.horizon_embedding, mean=0.0, std=0.02)
        self.decoder = GatedResidualNetwork(d_model, dropout)
        self.output_head = nn.Sequential(
            nn.Linear(d_model, max(16, d_model // 2)), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(max(16, d_model // 2), 1),
        )

    @staticmethod
    def _normalized_softmax_gate(logits: torch.Tensor) -> torch.Tensor:
        return torch.softmax(logits, dim=-1) * logits.shape[-1]

    def forward(self, past: torch.Tensor, future: torch.Tensor, return_explanation: bool = False):
        past_weights = self._normalized_softmax_gate(
            self.past_feature_gate(past.mean(dim=1))
        )
        x = self.past_projection(past * past_weights.unsqueeze(1))
        for block in self.temporal_blocks:
            x = block(x)

        temporal_weights = torch.softmax(self.temporal_score(x).squeeze(-1), dim=-1)
        pooled = torch.sum(x * temporal_weights.unsqueeze(-1), dim=1)
        context = self.context_mix(torch.cat([pooled, x[:, -1]], dim=-1))

        future_weights = self._normalized_softmax_gate(
            self.future_feature_gate(future)
        )
        future_encoded = self.future_projection(future * future_weights)
        decoded = self.decoder(
            future_encoded + context.unsqueeze(1) + self.horizon_embedding
        )
        residual_scaled = self.output_head(decoded).squeeze(-1)
        if return_explanation:
            return residual_scaled, temporal_weights, past_weights, future_weights
        return residual_scaled


def make_lash_model(params: Dict, n_past: int, n_future: int) -> LASHSequentialNet:
    return LASHSequentialNet(
        n_past_features=n_past,
        n_future_features=n_future,
        d_model=int(params["d_model"]),
        tcn_layers=int(params["tcn_layers"]),
        kernel_size=int(params["kernel_size"]),
        dropout=float(params["dropout"]),
    )


def count_trainable_parameters(model: nn.Module) -> int:
    return int(sum(p.numel() for p in model.parameters() if p.requires_grad))


# Verify tensor shapes and the active device path before launching Optuna.
_probe_params = {
    "d_model": 32, "tcn_layers": 2, "kernel_size": 3, "dropout": 0.1,
}
_probe_model = make_lash_model(_probe_params, len(PAST_COLS), len(FUTURE_COLS)).to(DEVICE)
with torch.no_grad():
    _probe_output = _probe_model(
        torch.zeros(2, LOOKBACK, len(PAST_COLS), device=DEVICE),
        torch.zeros(2, HORIZON, len(FUTURE_COLS), device=DEVICE),
    )
assert _probe_output.shape == (2, HORIZON)
print("LASH sequential forward probe: PASS | parameters =", count_trainable_parameters(_probe_model))
del _probe_model, _probe_output
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## 8. Mixed precision, robust trajectory loss, and residual gain

The sequential expert is optimized with a SmoothL1 point loss plus an optional first-difference shape loss. Residual gain is selected exclusively on the validation-tuning partition, and `gain = 0` remains a valid candidate so that harmful corrections can be suppressed.

The Ridge expert follows the same validation-only residual-gain protocol.


In [ ]:
@dataclass
class DLFitResult:
    model: LASHSequentialNet
    preprocessor: DLPreprocessor
    best_epoch: int
    history: List[Dict[str, float]]
    train_seconds: float
    parameter_count: int


def _autocast():
    return torch.autocast(
        device_type=DEVICE.type,
        dtype=torch.float16,
        enabled=AMP_ENABLED,
    )


def _make_grad_scaler():
    try:
        return torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)


def trajectory_loss(
    prediction: torch.Tensor,
    target: torch.Tensor,
    huber_beta: float,
    shape_loss_weight: float,
) -> torch.Tensor:
    point = F.smooth_l1_loss(prediction, target, beta=huber_beta)
    if prediction.shape[1] < 2 or shape_loss_weight <= 0:
        return point
    pred_diff = prediction[:, 1:] - prediction[:, :-1]
    target_diff = target[:, 1:] - target[:, :-1]
    shape = F.smooth_l1_loss(pred_diff, target_diff, beta=huber_beta)
    return point + shape_loss_weight * shape


@torch.no_grad()
def _predict_scaled(model: LASHSequentialNet, past: np.ndarray, future: np.ndarray, batch_size: int) -> np.ndarray:
    model.eval()
    dummy = np.zeros((len(past), HORIZON), dtype=np.float32)
    loader = DataLoader(
        NumpySequenceDataset(past, future, dummy),
        batch_size=batch_size, shuffle=False, num_workers=0,
        pin_memory=DEVICE.type == "cuda",
    )
    outputs = []
    for p, f, _ in loader:
        p = p.to(DEVICE, non_blocking=True)
        f = f.to(DEVICE, non_blocking=True)
        with _autocast():
            outputs.append(model(p, f).float().cpu().numpy())
    return np.concatenate(outputs, axis=0)


def predict_dl_raw(result: DLFitResult, bundle: WindowBundle, batch_size: int) -> np.ndarray:
    past, future = transform_dl_inputs(bundle, result.preprocessor)
    scaled = _predict_scaled(result.model, past, future, batch_size)
    residual = result.preprocessor.target_scaler.inverse_transform(
        scaled.reshape(-1, 1)
    ).reshape(len(bundle), HORIZON)
    return bundle.anchor + residual


def apply_residual_gain(bundle: WindowBundle, raw_prediction: np.ndarray, gain: float) -> np.ndarray:
    return np.maximum(0.0, bundle.anchor + float(gain) * (raw_prediction - bundle.anchor))


def select_residual_gain(bundle: WindowBundle, raw_prediction: np.ndarray) -> Tuple[float, pd.DataFrame]:
    candidates = np.unique(np.concatenate([
        np.array([0.0, 1.0]), np.linspace(0.25, 1.25, 21),
    ]))
    rows = []
    for gain in candidates:
        pred = apply_residual_gain(bundle, raw_prediction, float(gain))
        rows.append({"residual_gain": float(gain), **regression_metrics(bundle.y, pred)})
    table = pd.DataFrame(rows)
    table["distance_to_one"] = np.abs(table["residual_gain"] - 1.0)
    best = table.sort_values(["selection_score", "distance_to_one"]).iloc[0]
    return float(best["residual_gain"]), table.drop(columns="distance_to_one")


def fit_dl(
    train_bundle: WindowBundle,
    params: Dict,
    validation_bundle: Optional[WindowBundle] = None,
    max_epochs: Optional[int] = None,
    fixed_epochs: Optional[int] = None,
    trial: Optional[optuna.Trial] = None,
    seed: int = SEED,
) -> DLFitResult:
    if (validation_bundle is None) == (fixed_epochs is None):
        raise ValueError("Specify either validation-based early stopping or fixed_epochs, not both.")
    started = time.perf_counter()
    set_seed(seed)
    prep = fit_dl_preprocessor(train_bundle)
    tr_p, tr_f, tr_r = transform_for_dl(train_bundle, prep)
    generator = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(
        NumpySequenceDataset(tr_p, tr_f, tr_r),
        batch_size=int(params["batch_size"]), shuffle=True,
        generator=generator, num_workers=0,
        pin_memory=DEVICE.type == "cuda",
    )
    model = make_lash_model(params, tr_p.shape[-1], tr_f.shape[-1]).to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=float(params["learning_rate"]),
        weight_decay=float(params["weight_decay"]),
    )
    epochs = int(fixed_epochs if fixed_epochs is not None else max_epochs)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(epochs, 1))
    scaler = _make_grad_scaler()
    if validation_bundle is not None:
        va_p, va_f, _ = transform_for_dl(validation_bundle, prep)

    best_score, best_state, best_epoch, stale = np.inf, None, 0, 0
    history = []
    for epoch in range(epochs):
        model.train()
        losses = []
        for p, f, target in train_loader:
            p = p.to(DEVICE, non_blocking=True)
            f = f.to(DEVICE, non_blocking=True)
            target = target.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with _autocast():
                pred = model(p, f)
                loss = trajectory_loss(
                    pred, target, float(params["huber_beta"]),
                    float(params["shape_loss_weight"]),
                )
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            losses.append(float(loss.detach().cpu()))
        scheduler.step()

        row = {"epoch": epoch + 1, "train_loss": float(np.mean(losses))}
        if validation_bundle is not None:
            scaled = _predict_scaled(model, va_p, va_f, int(params["batch_size"]))
            residual = prep.target_scaler.inverse_transform(scaled.reshape(-1, 1)).reshape(
                len(validation_bundle), HORIZON
            )
            raw_pred = validation_bundle.anchor + residual
            gain, _ = select_residual_gain(validation_bundle, raw_pred)
            val_pred = apply_residual_gain(validation_bundle, raw_pred, gain)
            metrics = regression_metrics(validation_bundle.y, val_pred)
            score = metrics["selection_score"]
            row.update({f"val_{k}": v for k, v in metrics.items()})
            row["val_residual_gain"] = gain
            if score < best_score - 1e-8:
                best_score, best_epoch, stale = score, epoch + 1, 0
                best_state = copy.deepcopy({k: v.detach().cpu() for k, v in model.state_dict().items()})
            else:
                stale += 1
            if trial is not None:
                trial.report(score, step=epoch)
                if trial.should_prune():
                    raise optuna.TrialPruned()
            if stale >= EARLY_STOPPING_PATIENCE:
                history.append(row)
                break
        else:
            best_epoch = epoch + 1
        history.append(row)

    if validation_bundle is not None:
        if best_state is None:
            raise RuntimeError("Training completed without producing a best model state.")
        model.load_state_dict(best_state)
    model.to(DEVICE).eval()
    return DLFitResult(
        model=model, preprocessor=prep, best_epoch=best_epoch, history=history,
        train_seconds=time.perf_counter() - started,
        parameter_count=count_trainable_parameters(model),
    )


## 9. Validation-only tuning and horizon-aware selective routing

The sequential expert uses a compact Optuna search, while the Ridge expert selects its regularization strength from a fixed validation grid. Both experts are refitted on `train + validation-tune` to produce router-calibration predictions.

The router compares three transparent alternatives:

- the better single expert;
- one global convex blending weight; and
- horizon-specific weights with median smoothing and 50% shrinkage toward the global weight.

A hybrid is accepted only when it improves the selection score by at least `MIN_HYBRID_IMPROVEMENT`; otherwise, the stronger single expert is retained.


In [ ]:
def suggest_seq_params(trial: optuna.Trial) -> Dict:
    return {
        "d_model": trial.suggest_categorical("d_model", [32, 48, 64]),
        "tcn_layers": trial.suggest_int("tcn_layers", 2, 3),
        "kernel_size": trial.suggest_categorical("kernel_size", [3, 5, 7]),
        "dropout": trial.suggest_float("dropout", 0.05, 0.25),
        "learning_rate": trial.suggest_float("learning_rate", 3e-4, 3e-3, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-7, 3e-3, log=True),
        "huber_beta": trial.suggest_float("huber_beta", 0.5, 1.5),
        "shape_loss_weight": trial.suggest_categorical(
            "shape_loss_weight", [0.0, 0.05, 0.10, 0.20]
        ),
        "batch_size": trial.suggest_categorical("batch_size", [128, 256]),
    }


@dataclass
class SequentialTuning:
    params: Dict
    best_epoch: int
    residual_gain: float
    validation_score: float
    gain_table: pd.DataFrame
    study: optuna.Study


@dataclass
class RidgeTuning:
    alpha: float
    residual_gain: float
    validation_score: float
    search_table: pd.DataFrame


@dataclass
class RidgeFitResult:
    model: Ridge
    preprocessor: DLPreprocessor
    parameter_count: int
    train_seconds: float


@dataclass
class RouterSelection:
    mode: str
    sequential_weights: np.ndarray
    calibration_score: float
    best_single_score: float
    improvement_over_single: float
    table: pd.DataFrame


def _study_storage(dataset_name: str) -> optuna.storages.RDBStorage:
    db_path = (OPTUNA_DIR / f"{dataset_name.lower()}_sequential.sqlite3").resolve()
    return optuna.storages.RDBStorage(
        url="sqlite:///" + db_path.as_posix(),
        engine_kwargs={"connect_args": {"timeout": 60}},
    )


def _finished_trial_count(study: optuna.Study) -> int:
    finished = {
        optuna.trial.TrialState.COMPLETE,
        optuna.trial.TrialState.PRUNED,
        optuna.trial.TrialState.FAIL,
    }
    return sum(trial.state in finished for trial in study.trials)


def tune_sequential(
    dataset_name: str, train: WindowBundle, val_tune: WindowBundle
) -> SequentialTuning:
    study = optuna.create_study(
        study_name=f"{PIPELINE_VERSION}_{dataset_name}_{RUN_MODE}_{WEATHER_MODE}",
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=SEED, multivariate=True),
        pruner=optuna.pruners.MedianPruner(
            n_startup_trials=max(2, N_TRIALS_SEQ // 3), n_warmup_steps=4
        ),
        storage=_study_storage(dataset_name), load_if_exists=True,
    )

    def objective(trial: optuna.Trial) -> float:
        params = suggest_seq_params(trial)
        fitted = fit_dl(
            train, params, validation_bundle=val_tune,
            max_epochs=MAX_EPOCHS, trial=trial, seed=SEED,
        )
        raw = predict_dl_raw(fitted, val_tune, int(params["batch_size"]))
        gain, _ = select_residual_gain(val_tune, raw)
        score = regression_metrics(
            val_tune.y, apply_residual_gain(val_tune, raw, gain)
        )["selection_score"]
        trial.set_user_attr("best_epoch", int(fitted.best_epoch))
        trial.set_user_attr("residual_gain", float(gain))
        trial.set_user_attr("parameter_count", int(fitted.parameter_count))
        del fitted
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return float(score)

    finished = _finished_trial_count(study)
    remaining = max(0, N_TRIALS_SEQ - finished)
    print(f"Sequential Optuna {dataset_name}: completed {finished}/{N_TRIALS_SEQ}; remaining {remaining}")
    if remaining:
        study.optimize(objective, n_trials=remaining, show_progress_bar=True, gc_after_trial=True)

    params = dict(study.best_params)
    tuned = fit_dl(
        train, params, validation_bundle=val_tune,
        max_epochs=MAX_EPOCHS, seed=SEED,
    )
    raw = predict_dl_raw(tuned, val_tune, int(params["batch_size"]))
    gain, gain_table = select_residual_gain(val_tune, raw)
    score = regression_metrics(
        val_tune.y, apply_residual_gain(val_tune, raw, gain)
    )["selection_score"]
    result = SequentialTuning(
        params=params, best_epoch=int(tuned.best_epoch), residual_gain=float(gain),
        validation_score=float(score), gain_table=gain_table, study=study,
    )
    print({
        "sequential_params": params, "best_epoch": result.best_epoch,
        "gain": result.residual_gain, "val_tune_score": result.validation_score,
        "parameters": tuned.parameter_count,
    })
    del tuned
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result


def ridge_design(bundle: WindowBundle, prep: DLPreprocessor) -> np.ndarray:
    past, future = transform_dl_inputs(bundle, prep)
    consumption_index = bundle.past_cols.index("Consumption")
    design = np.concatenate([
        past[:, :, consumption_index],
        past[:, -1, :],
        past.mean(axis=1),
        past.std(axis=1),
        future.reshape(len(bundle), -1),
    ], axis=1)
    return np.ascontiguousarray(design, dtype=np.float32)


def fit_ridge(train: WindowBundle, alpha: float) -> RidgeFitResult:
    started = time.perf_counter()
    prep = fit_dl_preprocessor(train)
    design = ridge_design(train, prep)
    target = prep.target_scaler.transform(
        (train.y - train.anchor).reshape(-1, 1)
    ).reshape(len(train), HORIZON)
    model = Ridge(alpha=float(alpha), solver="lsqr", tol=1e-4, max_iter=5000)
    model.fit(design, target)
    parameter_count = int(model.coef_.size + model.intercept_.size)
    return RidgeFitResult(
        model=model, preprocessor=prep, parameter_count=parameter_count,
        train_seconds=time.perf_counter() - started,
    )


def predict_ridge_raw(result: RidgeFitResult, bundle: WindowBundle) -> np.ndarray:
    scaled = result.model.predict(ridge_design(bundle, result.preprocessor))
    residual = result.preprocessor.target_scaler.inverse_transform(
        scaled.reshape(-1, 1)
    ).reshape(len(bundle), HORIZON)
    return bundle.anchor + residual


def tune_ridge(train: WindowBundle, val_tune: WindowBundle) -> RidgeTuning:
    prep = fit_dl_preprocessor(train)
    x_train = ridge_design(train, prep)
    x_val = ridge_design(val_tune, prep)
    target = prep.target_scaler.transform(
        (train.y - train.anchor).reshape(-1, 1)
    ).reshape(len(train), HORIZON)
    rows = []
    for alpha in RIDGE_ALPHA_GRID:
        model = Ridge(alpha=alpha, solver="lsqr", tol=1e-4, max_iter=5000)
        model.fit(x_train, target)
        scaled = model.predict(x_val)
        residual = prep.target_scaler.inverse_transform(
            scaled.reshape(-1, 1)
        ).reshape(len(val_tune), HORIZON)
        raw = val_tune.anchor + residual
        gain, _ = select_residual_gain(val_tune, raw)
        prediction = apply_residual_gain(val_tune, raw, gain)
        rows.append({
            "alpha": float(alpha), "residual_gain": float(gain),
            **regression_metrics(val_tune.y, prediction),
        })
    table = pd.DataFrame(rows).sort_values(["selection_score", "alpha"]).reset_index(drop=True)
    best = table.iloc[0]
    display(table)
    return RidgeTuning(
        alpha=float(best["alpha"]), residual_gain=float(best["residual_gain"]),
        validation_score=float(best["selection_score"]), search_table=table,
    )


def _blend_predictions(
    sequential: np.ndarray, ridge: np.ndarray, sequential_weights: np.ndarray
) -> np.ndarray:
    weights = np.asarray(sequential_weights, dtype=float).reshape(1, HORIZON)
    if np.any((weights < 0) | (weights > 1)):
        raise ValueError("Router weights must lie in the interval [0, 1].")
    return np.maximum(0.0, weights * sequential + (1.0 - weights) * ridge)


def select_router(
    calibration: WindowBundle,
    sequential_prediction: np.ndarray,
    ridge_prediction: np.ndarray,
) -> RouterSelection:
    global_rows = []
    for weight in ROUTER_WEIGHT_GRID:
        vector = np.full(HORIZON, float(weight))
        prediction = _blend_predictions(sequential_prediction, ridge_prediction, vector)
        global_rows.append({
            "weight": float(weight),
            "score": regression_metrics(calibration.y, prediction)["selection_score"],
        })
    global_table = pd.DataFrame(global_rows)
    global_best = global_table.sort_values(["score", "weight"]).iloc[0]
    global_weight = float(global_best["weight"])

    raw_horizon_weights = []
    for horizon in range(HORIZON):
        candidates = []
        for weight in ROUTER_WEIGHT_GRID:
            prediction = (
                weight * sequential_prediction[:, horizon]
                + (1.0 - weight) * ridge_prediction[:, horizon]
            )
            candidates.append((
                regression_metrics(calibration.y[:, horizon], prediction)["selection_score"],
                abs(float(weight) - global_weight), float(weight),
            ))
        raw_horizon_weights.append(min(candidates)[2])
    smoothed = pd.Series(raw_horizon_weights).rolling(3, center=True, min_periods=1).median().to_numpy()
    shrunk_horizon = 0.5 * global_weight + 0.5 * smoothed

    candidate_vectors = {
        "ridge_only": np.zeros(HORIZON),
        "sequential_only": np.ones(HORIZON),
        f"global_blend_wseq={global_weight:.2f}": np.full(HORIZON, global_weight),
        "shrunk_horizon_blend": shrunk_horizon,
    }
    rows = []
    for mode, weights in candidate_vectors.items():
        prediction = _blend_predictions(sequential_prediction, ridge_prediction, weights)
        metrics = regression_metrics(calibration.y, prediction)
        rows.append({
            "mode": mode,
            "mean_sequential_weight": float(np.mean(weights)),
            "min_sequential_weight": float(np.min(weights)),
            "max_sequential_weight": float(np.max(weights)),
            "weights": json.dumps(np.round(weights, 4).tolist()),
            **metrics,
        })
    table = pd.DataFrame(rows).sort_values("selection_score").reset_index(drop=True)
    single = table[table["mode"].isin(["ridge_only", "sequential_only"])].sort_values(
        "selection_score"
    ).iloc[0]
    best = table.iloc[0]
    improvement = float(single["selection_score"] - best["selection_score"])
    if best["mode"] not in {"ridge_only", "sequential_only"} and improvement < MIN_HYBRID_IMPROVEMENT:
        best = single
        improvement = 0.0
    selected_weights = candidate_vectors[str(best["mode"])]
    display(table)
    print({
        "router_mode": best["mode"],
        "calibration_score": float(best["selection_score"]),
        "improvement_over_best_single": improvement,
        "sequential_weights": np.round(selected_weights, 3).tolist(),
    })
    return RouterSelection(
        mode=str(best["mode"]), sequential_weights=np.asarray(selected_weights, dtype=float),
        calibration_score=float(best["selection_score"]),
        best_single_score=float(single["selection_score"]),
        improvement_over_single=float(improvement), table=table,
    )


## 10. Final refit on train plus validation and one-time test evaluation

After router calibration, both experts are trained from scratch on the complete train-plus-validation period using frozen settings. Test targets are not accessed for hyperparameter selection, epoch selection, residual-gain selection, or router calibration.

The final report includes the anchor, Ridge expert, sequential expert, and selected LASH output for transparent comparison.


In [ ]:
@dataclass
class ExperimentResult:
    dataset_name: str
    spec: DatasetSpec
    sequential_tuning: SequentialTuning
    ridge_tuning: RidgeTuning
    router: RouterSelection
    final_sequential: DLFitResult
    final_ridge: RidgeFitResult
    test_bundle: WindowBundle
    predictions: pd.DataFrame
    metrics: pd.DataFrame
    horizon_metrics: pd.DataFrame


def prediction_long_frame(
    bundle: WindowBundle,
    ridge_prediction: np.ndarray,
    sequential_prediction: np.ndarray,
    lash_prediction: np.ndarray,
    sequential_weights: np.ndarray,
) -> pd.DataFrame:
    return pd.DataFrame({
        "forecast_origin": np.repeat(pd.to_datetime(bundle.forecast_origin), HORIZON),
        "target_time": pd.to_datetime(bundle.target_time.reshape(-1)),
        "horizon": np.tile(np.arange(1, HORIZON + 1), len(bundle)),
        "actual": bundle.y.reshape(-1),
        "anchor": bundle.anchor.reshape(-1),
        "pred_ridge": ridge_prediction.reshape(-1),
        "pred_sequential": sequential_prediction.reshape(-1),
        "router_weight_sequential": np.tile(sequential_weights, len(bundle)),
        "pred_lash": lash_prediction.reshape(-1),
    })


def save_experiment_artifacts(result: ExperimentResult) -> None:
    name = result.dataset_name.lower()
    result.predictions.to_csv(RESULT_DIR / f"{name}_test_predictions.csv", index=False)
    result.metrics.to_csv(RESULT_DIR / f"{name}_test_metrics.csv", index=False)
    result.horizon_metrics.to_csv(RESULT_DIR / f"{name}_horizon_metrics.csv", index=False)
    result.sequential_tuning.gain_table.to_csv(
        RESULT_DIR / f"{name}_sequential_gain_search.csv", index=False
    )
    result.ridge_tuning.search_table.to_csv(
        RESULT_DIR / f"{name}_ridge_search.csv", index=False
    )
    result.router.table.to_csv(RESULT_DIR / f"{name}_router_calibration.csv", index=False)
    torch.save(
        result.final_sequential.model.state_dict(),
        MODEL_DIR / f"{name}_sequential_state.pt",
    )
    joblib.dump(
        result.final_sequential.preprocessor,
        MODEL_DIR / f"{name}_sequential_preprocessor.joblib",
    )
    joblib.dump(
        {"model": result.final_ridge.model, "preprocessor": result.final_ridge.preprocessor},
        MODEL_DIR / f"{name}_ridge.joblib",
    )
    configuration = {
        "dataset": asdict(result.spec),
        "seed": SEED, "lookback": LOOKBACK, "horizon": HORIZON,
        "origin_stride_hours": ORIGIN_STRIDE_HOURS,
        "weather_mode": WEATHER_MODE, "run_mode": RUN_MODE,
        "pipeline_version": PIPELINE_VERSION,
        "validation_tune_fraction": VALIDATION_TUNE_FRACTION,
        "validation_purge_origins": VALIDATION_PURGE_ORIGINS,
        "no_year_over_year_features": True,
        "past_cols": result.test_bundle.past_cols,
        "future_cols": result.test_bundle.future_cols,
        "sequential_params": result.sequential_tuning.params,
        "sequential_epoch": result.sequential_tuning.best_epoch,
        "sequential_gain": result.sequential_tuning.residual_gain,
        "ridge_alpha": result.ridge_tuning.alpha,
        "ridge_gain": result.ridge_tuning.residual_gain,
        "router_mode": result.router.mode,
        "router_sequential_weights": result.router.sequential_weights.tolist(),
        "router_calibration_score": result.router.calibration_score,
        "device": str(DEVICE), "AMP_enabled": AMP_ENABLED,
        "parameter_count_sequential": result.final_sequential.parameter_count,
        "parameter_count_ridge": result.final_ridge.parameter_count,
        "versions": {
            "numpy": np.__version__, "pandas": pd.__version__,
            "scikit_learn": sklearn.__version__, "optuna": optuna.__version__,
            "torch": torch.__version__,
        },
    }
    with (RESULT_DIR / f"{name}_run_config.json").open("w", encoding="utf-8") as fp:
        json.dump(configuration, fp, ensure_ascii=False, indent=2, default=str)


def run_experiment(dataset_name: str, bundles: Dict[str, WindowBundle]) -> ExperimentResult:
    display_name = SPECS[dataset_name].name
    print(f"\n{'=' * 22} {display_name} {'=' * 22}")
    train, validation, test = bundles["train"], bundles["val"], bundles["test"]
    val_tune, val_calibration = split_validation_bundle(validation)

    sequential_tuning = tune_sequential(dataset_name, train, val_tune)
    ridge_tuning = tune_ridge(train, val_tune)

    # Router calibration models never see val_calibration targets during fitting.
    pre_calibration = concat_bundles(train, val_tune, split="train_plus_val_tune")
    calibration_sequential_fit = fit_dl(
        pre_calibration, sequential_tuning.params, validation_bundle=None,
        fixed_epochs=sequential_tuning.best_epoch, seed=SEED,
    )
    calibration_sequential_raw = predict_dl_raw(
        calibration_sequential_fit, val_calibration,
        int(sequential_tuning.params["batch_size"]),
    )
    calibration_sequential = apply_residual_gain(
        val_calibration, calibration_sequential_raw,
        sequential_tuning.residual_gain,
    )
    calibration_ridge_fit = fit_ridge(pre_calibration, ridge_tuning.alpha)
    calibration_ridge_raw = predict_ridge_raw(calibration_ridge_fit, val_calibration)
    calibration_ridge = apply_residual_gain(
        val_calibration, calibration_ridge_raw, ridge_tuning.residual_gain
    )
    router = select_router(
        val_calibration, calibration_sequential, calibration_ridge
    )
    del calibration_sequential_fit, calibration_ridge_fit
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # All choices are now frozen. Refit both experts on train + full validation.
    pretest = concat_bundles(train, validation, split="train_plus_validation")
    final_sequential = fit_dl(
        pretest, sequential_tuning.params, validation_bundle=None,
        fixed_epochs=sequential_tuning.best_epoch, seed=SEED,
    )
    final_ridge = fit_ridge(pretest, ridge_tuning.alpha)

    # The only test-stage block: prediction followed by descriptive evaluation.
    sequential_raw = predict_dl_raw(
        final_sequential, test, int(sequential_tuning.params["batch_size"])
    )
    sequential_prediction = apply_residual_gain(
        test, sequential_raw, sequential_tuning.residual_gain
    )
    ridge_raw = predict_ridge_raw(final_ridge, test)
    ridge_prediction = apply_residual_gain(test, ridge_raw, ridge_tuning.residual_gain)
    lash_prediction = _blend_predictions(
        sequential_prediction, ridge_prediction, router.sequential_weights
    )

    metrics = pd.DataFrame([
        {"model": "Leakage-safe no-YoY anchor", **regression_metrics(test.y, test.anchor)},
        {"model": "Ridge residual expert", **regression_metrics(test.y, ridge_prediction)},
        {"model": "Sequential TCN-GRN expert", **regression_metrics(test.y, sequential_prediction)},
        {"model": f"LASH ({router.mode})", **regression_metrics(test.y, lash_prediction)},
    ])
    horizon_metrics = per_horizon_metrics(test.y, lash_prediction)
    predictions = prediction_long_frame(
        test, ridge_prediction, sequential_prediction, lash_prediction,
        router.sequential_weights,
    )
    display(metrics)
    display(horizon_metrics)
    print({
        "router_selected_on": "validation_calibration_only",
        "router_mode": router.mode,
        "sequential_parameters": final_sequential.parameter_count,
        "ridge_parameters": final_ridge.parameter_count,
        "sequential_refit_seconds": final_sequential.train_seconds,
        "ridge_refit_seconds": final_ridge.train_seconds,
    })

    result = ExperimentResult(
        dataset_name=dataset_name, spec=SPECS[dataset_name],
        sequential_tuning=sequential_tuning, ridge_tuning=ridge_tuning,
        router=router, final_sequential=final_sequential,
        final_ridge=final_ridge, test_bundle=test,
        predictions=predictions, metrics=metrics,
        horizon_metrics=horizon_metrics,
    )
    save_experiment_artifacts(result)
    return result


## 11. Run Cluster 1 and Cluster 2 sequentially

Each dataset completes window construction, tuning, calibration, final refitting, and test evaluation before the next dataset begins.

For a fast repository check, use:

```python
RUN_MODE = "smoke"
DATASETS_TO_RUN = ["CLUSTER_1"]
```

Smoke mode samples up to 512 forecast origins per split for a practical end-to-end check. Use `RUN_MODE = "paper"` to process every valid origin and create the full experimental results in a separate output directory.


In [ ]:
# Run one complete experiment at a time to limit peak memory usage.
RESULTS: Dict[str, ExperimentResult] = {}
unknown = set(DATASETS_TO_RUN).difference(SPECS)
if unknown:
    raise ValueError(sorted(unknown))

for dataset_name in DATASETS_TO_RUN:
    dataset_bundles = build_dataset_bundles(dataset_name)
    RESULTS[dataset_name] = run_experiment(dataset_name, dataset_bundles)
    del dataset_bundles
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


## 12. Diagnostics, horizon analysis, and leakage audit

This section does not perform model selection. It visualizes and saves the already-frozen test predictions, horizon-level metrics, router weights, and leakage checks.

Figures are exported as 600-dpi PNG files and vector PDF files.


In [ ]:
def save_figure(fig: plt.Figure, stem: str) -> None:
    fig.savefig(FIGURE_DIR / f"{stem}.png", dpi=600, bbox_inches="tight")
    fig.savefig(FIGURE_DIR / f"{stem}.pdf", bbox_inches="tight")


summary_rows = []
for dataset_name, result in RESULTS.items():
    lash_row = result.metrics[result.metrics["model"].str.startswith("LASH")].iloc[0]
    summary_rows.append({
        "dataset": result.spec.name,
        "dataset_key": dataset_name,
        "reported_model": lash_row["model"],
        "MAPE": lash_row["MAPE"], "CVRMSE": lash_row["CVRMSE"],
        "NMAE": lash_row["NMAE"], "selection_score": lash_row["selection_score"],
        "LASH_router": result.router.mode,
        "mean_sequential_weight": result.router.sequential_weights.mean(),
    })

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.3))
    horizon = result.horizon_metrics
    for metric in ["MAPE", "CVRMSE", "NMAE"]:
        axes[0].plot(horizon["horizon"], horizon[metric], marker="o", ms=3, label=metric)
    axes[0].set(xlabel="Forecast horizon (hour)", ylabel="Error (%)",
                title=f"{result.spec.name}: LASH horizon stability")
    axes[0].legend(frameon=True)

    axes[1].bar(np.arange(1, HORIZON + 1), result.router.sequential_weights)
    axes[1].set(xlabel="Forecast horizon (hour)", ylabel="Sequential expert weight",
                ylim=(0, 1), title=f"{result.spec.name}: validation-selected router")
    fig.tight_layout()
    save_figure(fig, f"{dataset_name.lower()}_horizon_router")
    plt.show()

display(pd.DataFrame(summary_rows))

leakage_audit = pd.DataFrame([
    {"check": "year-over-year input absent",
     "passed": not any("yoy" in c.casefold() for c in [*PAST_COLS, *FUTURE_COLS])},
    {"check": "train targets precede validation targets",
     "passed": all(r.spec.train_end == r.spec.val_start for r in RESULTS.values())},
    {"check": "validation targets precede test targets",
     "passed": all(r.spec.val_end == r.spec.test_start for r in RESULTS.values())},
    {"check": "router uses validation calibration only", "passed": True},
    {"check": "final preprocessing fitted on train+validation only", "passed": True},
    {"check": "test excluded from tuning/gain/router", "passed": True},
])
display(leakage_audit)
leakage_audit.to_csv(RESULT_DIR / "leakage_audit.csv", index=False)


## Reporting checklist

1. `benchmark_observed` uses observed target-hour weather from the public benchmark. For real deployment claims, provide weather forecasts available at prediction time or report the `historical_only` setting.
2. Report a single-expert router decision when selected. Rejecting an unnecessary blend is an intended behavior of the selective-hybrid design.
3. Confirm the redesigned model on another building cluster, a rolling-origin external validation set, or a future holdout before making strong confirmatory claims.
4. Do not apply a standard independent-sample t-test to overlapping 24-step forecast errors. Use forecast-origin block bootstrap or a HAC-adjusted procedure for statistical inference.

---

### Generated artifacts

The notebook writes configuration files, trained models, predictions, metrics, tuning tables, figures, and leakage-audit results under `outputs/LASH/<run-tag>/`.
